# Notebook 06 — Data Agent: Student Persona

Configure and test an AI Data Agent scoped to individual students via Row-Level Security (RLS). Students can only see their own enrolment, exam, and financial data.

For this demo, we use a specific student: `STU-2021-0042`.

## Row-Level Security Reminder

The Student RLS role (configured in Notebook 03) filters `dim_student` by:
```dax
[email] = USERPRINCIPALNAME()
```
In this demo, we simulate RLS by filtering queries to a specific `student_key`.

## Student Agent System Prompt

```
You are a personal academic assistant for students at a Singapore university.
You help students understand their academic progress, grades, financial
obligations, and course history. You can only see data for the currently
logged-in student (enforced by Row-Level Security).

Currency is SGD. Grading follows the NUS 5.0 GPA scale (A+/A = 5.0 down
to F = 0.0). Each module is worth 4 Modular Credits (MC).
```

## Demo Student Context

In [ ]:
# Identify a demo student
demo_student = spark.sql("""
    SELECT student_key, student_id, first_name, last_name,
           email, domestic_international, enrolment_status,
           program_key, academic_year_start
    FROM university.dim_student
    WHERE student_id = 'STU-2021-0042'
""").collect()

if demo_student:
    row = demo_student[0]
    DEMO_SK = row['student_key']
    print(f"Demo Student: {row['first_name']} {row['last_name']} ({row['student_id']})")
    print(f"  Email: {row['email']}")
    print(f"  Status: {row['enrolment_status']}")
    print(f"  Type: {row['domestic_international']}")
    print(f"  student_key: {DEMO_SK}")
else:
    # Fallback: pick first active student
    row = spark.sql("SELECT student_key FROM university.dim_student WHERE enrolment_status = 'Active' LIMIT 1").collect()[0]
    DEMO_SK = row['student_key']
    print(f"Fallback demo student_key: {DEMO_SK}")

## Q1: What is my current GPA?

**Expected insight:** The student's overall average grade points.

In [ ]:
spark.sql(f"""
    SELECT ROUND(AVG(er.grade_points), 2) AS current_gpa,
           COUNT(*) AS total_assessments
    FROM university.fact_exam_results er
    WHERE er.student_key = {DEMO_SK}
""").show()

## Q2: How many modular credits have I completed so far?

**Expected insight:** MCs earned vs attempted.

In [ ]:
spark.sql(f"""
    SELECT SUM(fe.credit_points_earned) AS mcs_earned,
           SUM(fe.credit_points_attempted) AS mcs_attempted,
           COUNT(*) AS course_enrolments
    FROM university.fact_enrollments fe
    WHERE fe.student_key = {DEMO_SK}
""").show()

## Q3: Show my exam results for the last semester

**Expected insight:** Per-assessment scores for the most recent semester.

In [ ]:
spark.sql(f"""
    SELECT c.course_name, et.exam_type_name,
           er.score_percentage, er.grade_letter, er.grade_points,
           ap.period_label
    FROM university.fact_exam_results er
    JOIN university.dim_course c ON er.course_key = c.course_key
    JOIN university.dim_exam_type et ON er.exam_type_key = et.exam_type_key
    JOIN university.dim_academic_period ap ON er.academic_period_key = ap.academic_period_key
    WHERE er.student_key = {DEMO_SK}
      AND ap.academic_period_key = (SELECT MAX(academic_period_key) FROM university.dim_academic_period)
    ORDER BY c.course_name, et.exam_type_name
""").show(50, truncate=False)

## Q4: Do I have any outstanding fees?

**Expected insight:** Financial summary — charges, payments, and remaining balance in SGD.

In [ ]:
spark.sql(f"""
    SELECT
        ROUND(SUM(CASE WHEN transaction_type = 'Charge' THEN amount ELSE 0 END), 2) AS total_charges_sgd,
        ROUND(SUM(CASE WHEN transaction_type = 'Payment' THEN amount ELSE 0 END), 2) AS total_payments_sgd,
        ROUND(SUM(CASE WHEN transaction_type = 'Credit' THEN amount ELSE 0 END), 2) AS total_credits_sgd,
        ROUND(SUM(CASE WHEN transaction_type = 'Charge' THEN amount ELSE 0 END) -
              SUM(CASE WHEN transaction_type IN ('Payment', 'Credit') THEN amount ELSE 0 END), 2) AS outstanding_sgd
    FROM university.fact_financial_transactions
    WHERE student_key = {DEMO_SK}
""").show()

## Q5: What courses am I enrolled in this semester?

**Expected insight:** Current semester enrolments with course details.

In [ ]:
spark.sql(f"""
    SELECT c.course_id, c.course_name, fe.enrollment_status,
           fe.delivery_mode, ap.period_label
    FROM university.fact_enrollments fe
    JOIN university.dim_course c ON fe.course_key = c.course_key
    JOIN university.dim_academic_period ap ON fe.academic_period_key = ap.academic_period_key
    WHERE fe.student_key = {DEMO_SK}
      AND ap.academic_period_key = (SELECT MAX(academic_period_key) FROM university.dim_academic_period)
    ORDER BY c.course_name
""").show(truncate=False)

## Q6: How do my grades compare to the course average?

**Expected insight:** Student's score vs class average per course.

In [ ]:
spark.sql(f"""
    SELECT c.course_name,
           ROUND(AVG(CASE WHEN er.student_key = {DEMO_SK} THEN er.score_percentage END), 2) AS my_avg_score,
           ROUND(AVG(er.score_percentage), 2) AS class_avg_score,
           ROUND(AVG(CASE WHEN er.student_key = {DEMO_SK} THEN er.grade_points END), 2) AS my_gpa,
           ROUND(AVG(er.grade_points), 2) AS class_avg_gpa
    FROM university.fact_exam_results er
    JOIN university.dim_course c ON er.course_key = c.course_key
    WHERE er.course_key IN (SELECT DISTINCT course_key FROM university.fact_enrollments WHERE student_key = {DEMO_SK})
    GROUP BY c.course_name
    ORDER BY c.course_name
""").show(20, truncate=False)

## Q7: What is my grade distribution across all courses?

**Expected insight:** Count of each grade letter received.

In [ ]:
spark.sql(f"""
    SELECT er.grade_letter, COUNT(*) AS count
    FROM university.fact_exam_results er
    WHERE er.student_key = {DEMO_SK}
    GROUP BY er.grade_letter
    ORDER BY er.grade_letter
""").show()

## Q8: Have I received any scholarship credits?

**Expected insight:** Scholarship transactions for this student.

In [ ]:
spark.sql(f"""
    SELECT ft.transaction_id, ft_type.fee_type_name,
           ft.amount AS credit_sgd, ap.period_label
    FROM university.fact_financial_transactions ft
    JOIN university.dim_fee_type ft_type ON ft.fee_type_key = ft_type.fee_type_key
    JOIN university.dim_academic_period ap ON ft.academic_period_key = ap.academic_period_key
    WHERE ft.student_key = {DEMO_SK}
      AND ft.transaction_type = 'Credit'
    ORDER BY ap.academic_year, ap.semester
""").show(truncate=False)

## Q9: Show my academic progress by semester

**Expected insight:** GPA and MCs earned per semester over time.

In [ ]:
spark.sql(f"""
    SELECT ap.period_label,
           ROUND(AVG(er.grade_points), 2) AS semester_gpa,
           SUM(fe.credit_points_earned) AS mcs_earned,
           COUNT(DISTINCT fe.course_key) AS courses
    FROM university.fact_enrollments fe
    JOIN university.dim_academic_period ap ON fe.academic_period_key = ap.academic_period_key
    LEFT JOIN university.fact_exam_results er
        ON er.student_key = fe.student_key
        AND er.course_key = fe.course_key
        AND er.academic_period_key = fe.academic_period_key
    WHERE fe.student_key = {DEMO_SK}
    GROUP BY ap.period_label, ap.academic_year, ap.semester
    ORDER BY ap.academic_year, ap.semester
""").show(truncate=False)

## Q10: What courses have I failed or withdrawn from?

**Expected insight:** List of failed/withdrawn courses with details.

In [ ]:
spark.sql(f"""
    SELECT c.course_id, c.course_name, fe.enrollment_status,
           fe.course_final_grade_letter, ap.period_label
    FROM university.fact_enrollments fe
    JOIN university.dim_course c ON fe.course_key = c.course_key
    JOIN university.dim_academic_period ap ON fe.academic_period_key = ap.academic_period_key
    WHERE fe.student_key = {DEMO_SK}
      AND fe.enrollment_status IN ('Failed', 'Withdrawn')
    ORDER BY ap.academic_year, ap.semester
""").show(truncate=False)

## RLS Filter Demonstration

In production, RLS automatically filters data to the logged-in student. The queries above simulate this by explicitly filtering on `student_key`. In a real deployment, the student would never need to specify their ID — the `USERPRINCIPALNAME()` DAX filter handles this transparently.

In [ ]:
# Show what RLS would filter
spark.sql(f"""
    SELECT 'dim_student' AS table_name, COUNT(*) AS visible_rows FROM university.dim_student WHERE student_key = {DEMO_SK}
    UNION ALL
    SELECT 'fact_enrollments', COUNT(*) FROM university.fact_enrollments WHERE student_key = {DEMO_SK}
    UNION ALL
    SELECT 'fact_exam_results', COUNT(*) FROM university.fact_exam_results WHERE student_key = {DEMO_SK}
    UNION ALL
    SELECT 'fact_financial_transactions', COUNT(*) FROM university.fact_financial_transactions WHERE student_key = {DEMO_SK}
""").show()

## Demo Complete

This concludes the 6-notebook Fabric IQ Education Demo:

| Notebook | Purpose |
|---|---|
| 01 | Data generation and Lakehouse ingestion |
| 02 | Star schema Delta tables with DQ checks |
| 03 | Semantic model configuration guide |
| 04 | Copilot for Power BI demo |
| 05 | Data Agent — Staff persona |
| 06 | Data Agent — Student persona (with RLS) |

All data uses Singapore university context: SGD currency, NUS 5.0 GPA scale, Aug-start academic calendar, and MOE Singapore accreditation.